# Ajuste fino de um GPT-2 em português para o domínio de negócios

Este notebook prepara dados, adapta o modelo `pierreguillou/gpt2-small-portuguese` com LoRA e compara seu comportamento ao longo de três épocas de treinamento.

## 1. Verificação do ambiente

**O que esta célula faz:** mostra a versão do Python, o sistema operacional e a pasta de trabalho.

**Por que isso importa:** o ambiente funciona como a “bancada” do experimento. Registrar suas características ajuda a reproduzir o projeto e a entender diferenças entre computadores. Nesta execução foram usados Python 3.14.3, Windows e a pasta correta do projeto.

In [1]:
from pathlib import Path
import sys
import platform

print("Versão do Python:", sys.version)
print("Sistema operacional:", platform.system())
print("Pasta atual:", Path.cwd())

Versão do Python: 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Sistema operacional: Windows
Pasta atual: c:\Users\Pablo\Desktop\int-aprendizado-maquina


## 2. Instalação das bibliotecas de exploração

`pandas` organiza os dados em tabelas; `matplotlib` e `seaborn` servem para gráficos. O comando `%pip` instala os pacotes no mesmo ambiente usado pelo notebook.

**Resultado observado:** todas as bibliotecas já estavam instaladas. O aviso sobre uma versão nova do `pip` não é erro e não interfere no projeto.

In [2]:
%pip install pandas matplotlib seaborn

  Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 22.6 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 8.9 MB/s  0:00:00
Using cached kiwisolver-1.5.1-cp312-cp312-win_amd

## 3. Localização do conjunto de dados

Esta célula define o caminho relativo `data/negocios.jsonl` e confirma se o arquivo existe antes da leitura. É como conferir se um ingrediente está na bancada antes de começar uma receita.

**Resultado observado:** o arquivo foi encontrado. Caso contrário, o código listaria os arquivos JSON/JSONL próximos para ajudar a descobrir um caminho incorreto.

In [3]:
CAMINHO_DADOS = Path("./data/negocios.jsonl")

if CAMINHO_DADOS.exists():
    print("Arquivo encontrado:", CAMINHO_DADOS.resolve())
else:
    print("Arquivo não encontrado.")
    print("Caminho procurado:", CAMINHO_DADOS.resolve())

Arquivo encontrado: C:\Users\Pablo\Desktop\int-aprendizado-maquina\data\negocios.jsonl


## 4. Carregamento da base

`read_json(..., lines=True)` lê o formato JSONL, no qual cada linha representa um exemplo independente. Os dados são colocados em um `DataFrame`, uma tabela do pandas.

**Resultado observado:** a base possui 1.200 exemplos e 3 colunas. As três primeiras linhas são exibidas apenas como conferência visual.

In [4]:
import pandas as pd 

df = pd.read_json(CAMINHO_DADOS, lines=True)
print("Dimensões do DataFram:", df.shape)
display(df.head(3))

Dimensões do DataFram: (1200, 3)


,instruction,input,output
0,Componha um tweet sobre o investimento em crip...,,Investir em criptomoedas pode ser uma ótima ma...
1,Explique como as listas de tarefas podem ajuda...,,As listas de tarefas podem ajudar a aumentar a...
2,Explique a diferença entre atendimento ao clie...,,O atendimento antecipado ao cliente é focado e...


## 5. Estrutura e tipos das colunas

Esta inspeção confirma os nomes das colunas, a quantidade de valores preenchidos e seus tipos. O modelo espera `instruction` (pedido), `input` (contexto opcional) e `output` (resposta esperada).

**Resultado observado:** todas as colunas existem, são texto e têm 1.200 valores não nulos.

In [5]:
print("Colunas do DataFrame:", df.columns.tolist())
print()
df.info()

Colunas do DataFrame: ['instruction', 'input', 'output']

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   instruction  1200 non-null   str  
 1   input        1200 non-null   str  
 2   output       1200 non-null   str  
dtypes: str(3)
memory usage: 28.3 KB


## 6. Contagem de valores nulos e textos vazios

Um valor **nulo** significa ausência de dado; um texto **vazio** existe, mas não contém caracteres úteis. A diferença é parecida com “não há caixa” versus “há uma caixa, mas ela está vazia”.

Esta célula mede ambos os casos nas três colunas. O resultado mostra 949 `input` vazios, mas nenhuma instrução ou resposta ausente. Isso é aceitável porque o contexto foi definido como opcional.

In [6]:
colunas_texto = ["instruction", "input", "output"]

resumo_qualidade = pd.DataFrame({
    "nulos": df[colunas_texto].isna().sum(),
    "vazios": df[colunas_texto].apply(
        lambda coluna: coluna.str.strip().eq("").sum()
    )
})

resumo_qualidade

,nulos,vazios
instruction,0,0
input,0,949
output,0,0


## 7. Proporção de exemplos sem contexto

Esta célula transforma a contagem anterior em porcentagem para facilitar a interpretação.

**Resultado observado:** 79,08% dos exemplos não têm contexto adicional. Portanto, o formato de treinamento precisa funcionar tanto com perguntas isoladas quanto com perguntas acompanhadas de contexto.

In [7]:
proporcao_input_vazio = (
    resumo_qualidade.loc["input", "vazios"] / len(df)
) * 100

print(f"Inputs vazios: {proporcao_input_vazio:.2f}%")

Inputs vazios: 79.08%


## 8. Busca por duplicatas

Exemplos repetidos recebem peso extra no aprendizado, como uma frase ouvida muitas vezes. Isso pode estimular memorização e produzir uma avaliação otimista demais.

**Resultado observado:** não existem linhas inteiramente duplicadas nem instruções duplicadas. Não foi necessário remover registros por esse motivo.

In [8]:
duplicatas_exatas = df.duplicated().sum()

instrucoes_duplicadas = df.duplicated(
    subset=["instruction"]
).sum()

print("Linhas completamente duplicadas:", duplicatas_exatas)
print("Instruções duplicadas:", instrucoes_duplicadas)

Linhas completamente duplicadas: 0
Instruções duplicadas: 0


## 9. Medição do tamanho dos textos em caracteres

O código conta caracteres em cada coluna e acrescenta as medidas à tabela original. Caracteres não são iguais a tokens, mas funcionam como uma primeira estimativa do tamanho dos exemplos.

**Por que isso importa:** textos muito longos podem ultrapassar o limite de entrada do modelo e aumentar o consumo de memória.

In [9]:
comprimentos = df[colunas_texto].apply(
    lambda coluna: coluna.str.len()
)

comprimentos.columns = [
    "instruction_chars",
    "input_chars",
    "output_chars"
]

comprimentos.head()

,instruction_chars,input_chars,output_chars
0,55,0,158
1,75,0,347
2,71,0,638
3,65,60,227
4,56,0,314


## 10. Resumo estatístico dos comprimentos

Aqui são calculados mínimo, média, mediana, percentis e máximo. **Percentil** é um ponto de corte: por exemplo, o percentil 95 indica um tamanho maior ou igual ao de 95% dos registros.

**Resultado observado:** a maior resposta tem 789 caracteres e a maior instrução, 204. Um output com apenas 1 caractere merece atenção na análise qualitativa, pois pode ser legítimo em tarefas de classificação, mas também pode indicar baixa qualidade.

In [10]:
comprimentos.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T.round(2)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
instruction_chars,1200.0,72.64,24.50,22.0,55.0,70.0,84.00,101.0,119.00,151.08,204.0
input_chars,1200.0,12.48,32.41,0.0,0.0,0.0,0.00,52.0,85.05,141.02,335.0
output_chars,1200.0,331.35,205.00,1.0,134.0,344.0,511.75,604.0,633.05,690.01,789.0


## 11. Amostra aleatória para análise qualitativa

Números revelam padrões gerais, mas não mostram se o conteúdo faz sentido. Por isso, esta célula sorteia cinco exemplos com `random_state=42`, que torna o sorteio reproduzível.

O objetivo é conferir clareza das instruções, coerência das respostas e adequação ao domínio de negócios.

In [11]:
pd.set_option("display.max_colwidth", None)

amostra_aleatoria = df.sample(
    n=5,
    random_state=42
)

display(amostra_aleatoria)

,instruction,input,output
1178,Identifique cinco qualidades de um bom representante de atendimento ao cliente.,,"Aqui estão cinco qualidades de um bom representante de atendimento ao cliente: - Possuir boa comunicação e habilidades interpessoais - Ser paciente, amigável e profissional - Ter uma mentalidade de resolução de problemas - Ser adaptável e capaz de tomar decisões rápidas - Ser orientado para os detalhes e organizado - Ser capaz de trabalhar bem sob pressão - Ser experiente em tecnologia e experiente sobre os produtos e serviços - Ter uma abordagem amigável e empática em relação aos clientes - Possuir boas habilidades de resolução de conflitos - Ter uma capacidade de lidar com as reclamações dos clientes de forma positiva"
865,Quais são as características de uma ideia de negócio de sucesso?,,"Uma ideia de negócio bem-sucedida deve ter essas características: uma proposta de venda única, a capacidade de atender a uma necessidade de mercado, potencial de escalabilidade, eficiência de capital e potencial de mercado realista. Além disso, a empresa deve ser capaz de ser executada dentro de um determinado orçamento e prazo e deve ser capaz de fornecer um retorno sobre o investimento para seus investidores."
101,"A Amazon é uma empresa Fortune 500. Classifique-a como uma empresa de fabricação, empresa de serviços ou empresa de tecnologia.",,Empresa de tecnologia.
439,Compare o custo e a eficiência de uma turbina eólica com um painel solar,,"Em geral, as turbinas eólicas são mais econômicas do que os painéis solares. As turbinas eólicas geralmente têm uma taxa mais alta de retorno sobre o investimento, pois exigem menos investimento adiantado. Além disso, elas podem gerar energia mesmo à noite ou em condições turvas, enquanto os painéis solares exigem luz solar direta para serem eficazes. Por outro lado, os painéis solares são tipicamente mais baratos de instalar e menos propensos a quebra."
58,Crie um tópico de pesquisa relacionado ao aumento da fidelidade do cliente.,,"O tópico de pesquisa pode se concentrar na eficácia dos programas de fidelidade no aumento da fidelidade do cliente. A pesquisa pode analisar os diferentes tipos de programas de fidelidade usados pelas empresas e as estratégias usadas pelas empresas para tornar os programas de fidelidade atraentes para os clientes. Além disso, a pesquisa também pode analisar o comportamento do cliente antes, durante e depois dos programas de fidelidade, como a taxa de retenção do cliente, a satisfação do cliente e a defesa do cliente."


## 12. Inspeção das respostas mais curtas

Esta célula ordena a base pelo tamanho do `output` e exibe os cinco menores. Respostas curtas não são automaticamente erros: uma nota, classe ou valor numérico pode ser a resposta correta.

A revisão manual evita remover exemplos válidos apenas porque eles fogem da média.

In [12]:
df_com_tamanho = df.assign(
    output_chars=df["output"].str.len()
)

respostas_mais_curtas = df_com_tamanho.nsmallest(
    5,
    "output_chars"
)[
    ["instruction", "input", "output", "output_chars"]
]

display(respostas_mais_curtas)

,instruction,input,output,output_chars
768,"Dada uma avaliação do cliente, marque o sentimento geral da revisão, com 0 sendo muito negativo e 10 sendo muito positivo.",Estou decepcionado com o atraso e isso arruinou meus planos.,2,1
935,Classifique a seguinte consulta como um dos seguintes tópicos: 1) oferta de produto; 2) pagamento e faturamento; 3) entrega e envio; 4) conta e assinatura; 5) problema técnico.,"Estou tentando conectar minha conta ao site, mas continuo recebendo uma mensagem de erro.",5,1
52,"Encontre o preço total para os itens na lista dada, uma vez que cada item na lista é de US $ 10.","Lista: laptop, celular, mouse de computador",30,2
770,"Você é solicitado a avaliar o seguinte restaurante com base em uma escala de 1 a 10, com 10 sendo o mais alto. ""O restaurante tem uma excelente atmosfera, funcionários simpáticos e comida deliciosa.""",,10.,3
933,"Calcule o custo total dos itens abaixo, supondo que cada item seja comprado nas quantidades listadas.","Apple - 2 kg a US $ 4 / kg; Laranja - 5 a US $ 1,50 cada",$14,3


## 13. Inspeção das respostas mais longas

Agora são mostradas as cinco respostas maiores. A finalidade é verificar se textos extensos continuam relevantes e bem formados e se não dominam desnecessariamente o treinamento.

Essa inspeção também ajuda a escolher o limite de tokens usado mais adiante.

In [13]:
respostas_mais_longas = df_com_tamanho.nlargest(
    5,
    "output_chars"
)[
    ["instruction", "input", "output", "output_chars"]
]

display(respostas_mais_longas)

,instruction,input,output,output_chars
357,Desenvolver um plano para monitorar o envolvimento do cliente em plataformas de mídia social,,"Um plano para monitorar o envolvimento do cliente em plataformas de mídia social deve incluir as seguintes etapas: 1. Identifique as plataformas de mídia social mais relevantes para o envolvimento do cliente. 2. Configure ferramentas de monitoramento que possam rastrear conversas e engajamento do cliente nas plataformas de mídia social. 3. Atribua indicadores de desempenho chave relevantes (KPIs) que possam ser rastreados para medir o envolvimento do cliente. 4. Desenvolva um processo para analisar e interpretar os dados coletados das ferramentas de monitoramento. 5. Desenvolva um plano para agir sobre os dados coletados, como, ajustar, como estratégias de mensagens, como, ajustar, ajustar, ajustar, ajustar, ajustar, ajustar, ajustar, ajustar e gerenciar e gerenciar e gerenciar.",789
610,Crie um novo plano de implementação para reduzir o preço do transporte nos Estados Unidos.,,"Um plano de implementação para reduzir o preço do transporte nos Estados Unidos deve incluir as seguintes medidas. Em primeiro lugar, o governo federal deve apoiar ativamente as iniciativas de transporte público e fornecer incentivos financeiros para sistemas de transporte público. Em segundo lugar, o governo deve investir no desenvolvimento de veículos mais eficientes e fontes de energia alternativas, como veículos elétricos ou híbridos. Em terceiro lugar, as empresas de automóveis privadas devem ser incentivadas a implementar incentivos para a compra de veículos eficientes em termos de combustível ou veículos movidos por fontes de energia alternativas. Em quarto lugar, os estados devem ser incentivados para fornecer melhores subsídios de transporte.",761
946,Delinear os benefícios de uma empresa usando o serviço ao cliente orientado por IA,,"Uma empresa que faz uso do atendimento ao cliente orientado por IA se beneficiará de várias maneiras. Em primeiro lugar, as plataformas de atendimento ao cliente orientadas por IA podem fornecer experiências personalizadas aos clientes, utilizando os dados do cliente para entender melhor suas necessidades e preferências. Em segundo lugar, a IA permite que os agentes de atendimento ao cliente lidem com mais rapidez e eficiência com solicitações e consultas do cliente. Em terceiro lugar, as plataformas de atendimento ao cliente orientadas por IA podem automatizar processos como responder a e-mails e chats, além de clientes de entrada que também podem ajudar os agentes apropriados. Por fim, as tendências de serviço de IA podem ser usadas.",745
1058,Descreva um cenário em que uma empresa pode utilizar o feedback do cliente para melhorar o atendimento ao cliente.,,"As empresas podem utilizar o feedback do cliente para melhorar o atendimento ao cliente, pesquisando periodicamente os clientes sobre sua experiência. As pesquisas podem ser conduzidas através de chamadas de atendimento ao cliente, e-mails ou questionários on-line. Este feedback do cliente pode ser usado para identificar áreas de serviço que os clientes acham falta ou áreas de melhoria que podem ser feitas. As empresas podem então criar tarefas e objetivos para resolver quaisquer problemas de atendimento ao cliente identificados. Além disso, o feedback do cliente pode ser usado para moldar os treinamentos de atendimento ao cliente, garantir que a equipe do atendimento ao cliente seja devidamente treinada.",714
385,Crie 3 estratégias para prever as tendências do consumidor.,,"1. Analisando dados do consumidor: Reunir pontos de dados do consumidor, como comportamento de compra e respostas de pesquisa, pode dar uma visão sobre as tendências do consumidor. Esta informação pode ser usada para prever tendências futuras. 2. Pesquisas e grupos focais: Pesquisas e grupos focais podem dar uma visão sobre o comportamento do consumidor em potencial. Este feedback pode ajudar a prever as tendências do consumidor e descobrir potenciais

## 14. Padronização do texto de treinamento

A função `formatar_exemplo` converte cada linha em uma sequência com marcadores: `Pergunta`, `Contexto` (somente quando existir) e `Resposta`.

Esses marcadores funcionam como etiquetas em caixas: ajudam o modelo a distinguir o pedido, a informação auxiliar e aquilo que ele deve aprender a produzir. `strip()` remove espaços extras nas extremidades.

In [14]:
def formatar_exemplo(linha):
    instrucao = linha["instruction"].strip()
    contexto = linha["input"].strip()
    resposta = linha["output"].strip()

    partes = [
        f"### Pergunta:\n{instrucao}"
    ]

    if contexto:
        partes.append(
            f"### Contexto:\n{contexto}"
        )

    partes.append(
        f"### Resposta:\n{resposta}"
    )

    return "\n\n".join(partes)

## 15. Instalação da biblioteca para dividir os dados

O `scikit-learn` fornece `train_test_split`, usado para criar os conjuntos de treino, validação e teste.

**Aviso registrado:** a mensagem “may need to restart the kernel” é preventiva e comum após instalações. Como a importação seguinte funcionou, não houve conflito nesta execução. O aviso de atualização do `pip` também não afeta o modelo.

In [15]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 16. Divisão em treino, validação e teste

Os dados são separados em 80% para treino, 10% para validação e 10% para teste. O treino ensina; a validação funciona como um simulado durante as decisões; o teste deve funcionar como a prova final.

`random_state=42` reproduz a mesma divisão. O resultado foi 960 exemplos de treino, 120 de validação e 120 de teste.

In [16]:
from sklearn.model_selection import train_test_split

df_treino, df_temporario = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

df_validacao, df_teste = train_test_split(
    df_temporario,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Treino:", len(df_treino))
print("Validação:", len(df_validacao))
print("Teste:", len(df_teste))
print("Total:", len(df_treino) + len(df_validacao) + len(df_teste))

Treino: 960
Validação: 120
Teste: 120
Total: 1200


## 17. Verificação de vazamento entre conjuntos

**Vazamento de dados** ocorre quando um exemplo usado para ensinar também aparece na avaliação — como receber na prova uma questão já resolvida durante o estudo. Isso faz o desempenho parecer melhor do que realmente é.

O cruzamento dos índices encontrou zero sobreposição e confirmou 1.200 índices diferentes.

In [17]:
indices_treino = set(df_treino.index)
indices_validacao = set(df_validacao.index)
indices_teste = set(df_teste.index)

print("Repetidos entre treino e validação:", len(indices_treino & indices_validacao))
print("Repetidos entre treino e teste:", len(indices_treino & indices_teste))
print("Repetidos entre validação e teste:", len(indices_validacao & indices_teste))
print("Total de índices diferentes:", len(indices_treino | indices_validacao | indices_teste))

Repetidos entre treino e validação: 0
Repetidos entre treino e teste: 0
Repetidos entre validação e teste: 0
Total de índices diferentes: 1200


## 18. Distribuição dos exemplos com contexto

Esta célula mede se a presença de contexto ficou razoavelmente distribuída após a divisão aleatória.

**Resultado observado:** treino 21,15%, validação 22,50% e teste 17,50%. Há uma pequena diferença no teste, esperada em divisões aleatórias pequenas; não é um erro, mas uma divisão estratificada poderia equilibrar exatamente essa característica.

In [18]:
conjuntos = {
    "Treino": df_treino,
    "Validação": df_validacao,
    "Teste": df_teste
}

for nome, dados in conjuntos.items():
    possui_contexto = dados["input"].str.strip() != ""
    quantidade = possui_contexto.sum()
    percentual = quantidade / len(dados) * 100
    print(f"{nome}: {quantidade} exemplos com contexto ({percentual:.2f}%)")

Treino: 203 exemplos com contexto (21.15%)
Validação: 27 exemplos com contexto (22.50%)
Teste: 21 exemplos com contexto (17.50%)


## 19. Carregamento do tokenizador

O **tokenizador** quebra texto em unidades numéricas chamadas tokens, como quem separa uma frase em peças que o modelo consegue processar. O tokenizador deve ser o mesmo usado pelo modelo-base.

Como o GPT-2 não possui `pad_token` próprio, o código reutiliza o token de fim de texto (`EOS`) para preencher lotes. Ambos ficaram com ID 0; mais adiante, a máscara impede que esse preenchimento conte na loss.

**Avisos registrados:** a ausência de `ipywidgets` afeta apenas a aparência da barra de progresso no Jupyter. O acesso sem `HF_TOKEN` pode reduzir limites ou velocidade de download no Hugging Face Hub, mas não altera os pesos carregados.

In [19]:
from transformers import AutoTokenizer

MODELO_BASE = "pierreguillou/gpt2-small-portuguese"
tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token

print("Tipo:", type(tokenizador).__name__)
print("Vocabulário:", len(tokenizador))
print("EOS:", tokenizador.eos_token, tokenizador.eos_token_id)
print("PAD:", tokenizador.pad_token, tokenizador.pad_token_id)

c:\Users\Pablo\Desktop\int-aprendizado-maquina\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tipo: GPT2Tokenizer
Vocabulário: 50257
EOS: <|endoftext|> 0
PAD: <|endoftext|> 0


## 20. Verificação da GPU

GPU é um processador muito eficiente em cálculos paralelos, usados intensamente no treinamento de redes neurais. CUDA é a tecnologia que permite ao PyTorch utilizar GPUs NVIDIA.

**Resultado observado:** a RTX 3050 Laptop foi reconhecida, com 4 GB de VRAM. Como essa memória é limitada, o projeto adota batch pequeno, precisão de 16 bits e LoRA.

In [20]:
import torch

print("Versão do PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Versão do PyTorch: 2.14.0+cu126
CUDA do PyTorch: 12.6
GPU disponível: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


## 21. Teste de tokenização e reconstrução

Um exemplo é formatado, finalizado com `EOS`, convertido em 66 tokens e depois decodificado novamente. A máscara de atenção contém apenas 1 porque ainda não há preenchimento neste exemplo isolado.

**Resultado observado:** a reconstrução foi idêntica ao texto original e o último ID foi 0, confirmando o token de fim de texto.

In [21]:
texto_treino = formatar_exemplo(df.loc[768]) + tokenizador.eos_token

codificacao = tokenizador(
    texto_treino,
    add_special_tokens=False
)

ids = codificacao["input_ids"]
mascara = codificacao["attention_mask"]

print("Caracteres:", len(texto_treino))
print("Tokens:", len(ids))
print("Valores da máscara:", set(mascara))
print("Último ID:", ids[-1])

texto_recuperado = tokenizador.decode(ids, skip_special_tokens=False)
print("Reconstrução correta:", texto_recuperado == texto_treino)

Caracteres: 242
Tokens: 66
Valores da máscara: {1}
Último ID: 0
Reconstrução correta: True


## 22. Comprimento real em tokens e escolha de `MAX_LENGTH`

Tokens, e não caracteres, determinam o espaço ocupado na entrada do modelo. Esta célula mede todos os 960 textos de treino antes de escolher o limite.

**Resultado observado:** o maior exemplo tem 199 tokens; nenhum ultrapassa 256. Assim, `MAX_LENGTH = 256` preserva toda a base atual e deixa uma margem de segurança. Embora 296 exemplos passem de 128 tokens, eles não serão cortados.

In [22]:
textos_treino = df_treino.apply(
    lambda linha: formatar_exemplo(linha) + tokenizador.eos_token,
    axis=1
)

comprimentos_treino = textos_treino.apply(
    lambda texto: len(
        tokenizador(texto, add_special_tokens=False)["input_ids"]
    )
)

print("Quantidade de textos:", len(textos_treino))
print("Menor:", comprimentos_treino.min())
print("Média:", round(comprimentos_treino.mean(), 2))
print("Mediana:", comprimentos_treino.median())
print("Maior:", comprimentos_treino.max())
print("Acima de 128:", (comprimentos_treino > 128).sum())
print("Acima de 256:", (comprimentos_treino > 256).sum())
MAX_LENGTH = 256

Quantidade de textos: 960
Menor: 30
Média: 103.24
Mediana: 105.0
Maior: 199
Acima de 128: 296
Acima de 256: 0


## 23. Tokenização do conjunto de treino

Cada texto vira `input_ids` (os números dos tokens) e `attention_mask` (indicação de quais posições são conteúdo real). `truncation=True` protege contra exemplos acima do limite, embora a medição anterior mostre que nenhum será truncado nesta base.

O padding ainda não é aplicado: cada exemplo conserva seu comprimento natural para economizar memória.

In [23]:
dados_treino_tokenizados = []

for texto in textos_treino:
    codificacao = tokenizador(
        texto,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        add_special_tokens=False
    )

    dados_treino_tokenizados.append({
        "input_ids": codificacao["input_ids"],
        "attention_mask": codificacao["attention_mask"]
    })

print("Exemplos tokenizados:", len(dados_treino_tokenizados))
print("Campos:", list(dados_treino_tokenizados[0].keys()))
print("Tokens do primeiro:", len(dados_treino_tokenizados[0]["input_ids"]))
print("Máscara do primeiro:", set(dados_treino_tokenizados[0]["attention_mask"]))

Exemplos tokenizados: 960
Campos: ['input_ids', 'attention_mask']
Tokens do primeiro: 145
Máscara do primeiro: {1}


## 24. Função de padding dinâmico e criação das labels

**Padding** é um preenchimento temporário que iguala o comprimento dos exemplos de um mesmo lote, como completar caixas de tamanhos diferentes até terem a mesma altura. Ele é dinâmico porque usa apenas o tamanho do maior exemplo do lote.

As `labels` representam os tokens que o modelo deve prever. Posições de padding recebem `-100`, valor que instrui a função de loss a ignorá-las; assim, o modelo não é punido por preenchimentos artificiais.

In [24]:
def montar_lote(exemplos):
    lote = tokenizador.pad(
        exemplos,
        padding=True,
        return_tensors="pt"
    )

    labels = lote["input_ids"].clone()
    labels[lote["attention_mask"] == 0] = -100
    lote["labels"] = labels

    return lote

## 25. Teste do montador de lotes

Antes do treinamento, quatro exemplos passam pela função `montar_lote`. Esse teste confirma formatos, máscaras e labels.

**Resultado observado:** os comprimentos diferentes foram alinhados em 145 posições; 165 paddings foram corretamente marcados com `-100`. É uma checagem preventiva contra perdas calculadas sobre conteúdo falso.

In [25]:
exemplos_do_lote = dados_treino_tokenizados[:4]
tamanhos_antes = [len(exemplo["input_ids"]) for exemplo in exemplos_do_lote]
lote_teste = montar_lote(exemplos_do_lote)

print("Tamanhos antes:", tamanhos_antes)
print("Campos:", list(lote_teste.keys()))
print("Formato:", lote_teste["input_ids"].shape)
print("Tokens verdadeiros:", lote_teste["attention_mask"].sum(dim=1).tolist())
print("Paddings:", (lote_teste["attention_mask"] == 0).sum(dim=1).tolist())
print("Labels ignoradas:", (lote_teste["labels"] == -100).sum().item())

Tamanhos antes: [145, 54, 106, 110]
Campos: ['input_ids', 'attention_mask', 'labels']
Formato: torch.Size([4, 145])
Tokens verdadeiros: [145, 54, 106, 110]
Paddings: [0, 91, 39, 35]
Labels ignoradas: 165


## 26. Criação do DataLoader de treino

O `DataLoader` entrega os dados ao modelo em pequenos lotes. `BATCH_SIZE = 2` significa dois exemplos por passo; é uma escolha conservadora para os 4 GB de VRAM. `shuffle=True` muda a ordem dos exemplos a cada época.

Com 960 exemplos, o resultado é 480 lotes por época.

In [26]:
from torch.utils.data import DataLoader

BATCH_SIZE = 2

carregador_treino = DataLoader(
    dados_treino_tokenizados,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=montar_lote
)

primeiro_lote = next(iter(carregador_treino))

print("Campos:", list(primeiro_lote.keys()))
print("Formato:", primeiro_lote["input_ids"].shape)
print("Tokens verdadeiros:", primeiro_lote["attention_mask"].sum(dim=1).tolist())
print("Paddings:", (primeiro_lote["attention_mask"] == 0).sum(dim=1).tolist())
print("Total de lotes:", len(carregador_treino))

Campos: ['input_ids', 'attention_mask', 'labels']
Formato: torch.Size([2, 117])
Tokens verdadeiros: [72, 117]
Paddings: [45, 0]
Total de lotes: 480


## 27. Preparação da validação

O conjunto de validação passa exatamente pela mesma formatação e tokenização do treino, mas usa `shuffle=False`, pois não há necessidade de embaralhar durante a medição.

Foram preparados 120 exemplos em 60 lotes. Manter o mesmo pré-processamento evita comparar conjuntos tratados de formas diferentes.

In [27]:
textos_validacao = df_validacao.apply(
    lambda linha: formatar_exemplo(linha) + tokenizador.eos_token,
    axis=1
)

dados_validacao_tokenizados = []

for texto in textos_validacao:
    codificacao = tokenizador(
        texto,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        add_special_tokens=False
    )

    dados_validacao_tokenizados.append({
        "input_ids": codificacao["input_ids"],
        "attention_mask": codificacao["attention_mask"]
    })

carregador_validacao = DataLoader(
    dados_validacao_tokenizados,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=montar_lote
)

print("Exemplos de validação:", len(dados_validacao_tokenizados))
print("Lotes de validação:", len(carregador_validacao))

Exemplos de validação: 120
Lotes de validação: 60


## 28. Testes de consistência da preparação

Os `asserts` são alarmes: interrompem a execução se quantidades ou hiperparâmetros importantes forem diferentes do esperado. Isso detecta alterações acidentais antes de um treinamento caro.

**Resultado observado:** todas as verificações passaram e a GPU continuou disponível. Não houve erro nesta etapa.

In [28]:
assert len(df_treino) == 960
assert len(df_validacao) == 120
assert len(df_teste) == 120

assert len(dados_treino_tokenizados) == 960
assert len(dados_validacao_tokenizados) == 120

assert len(carregador_treino) == 480
assert len(carregador_validacao) == 60

assert MAX_LENGTH == 256
assert BATCH_SIZE == 2

print("Preparação dos dados concluída com sucesso!")
print("GPU disponível:", torch.cuda.is_available())

Preparação dos dados concluída com sucesso!
GPU disponível: True


## 29. Instalação do PEFT e Accelerate

PEFT reúne técnicas para adaptar modelos treinando poucos parâmetros; LoRA será a técnica escolhida. Accelerate auxilia a execução eficiente em CPU/GPU.

Nesta execução os pacotes foram instalados com sucesso. O aviso de atualização do `pip` é informativo.

In [29]:
%pip install peft accelerate

Note: you may need to restart the kernel to use updated packages.


## 30. Registro das versões de PEFT e Accelerate

Esta célula confirma que as bibliotecas podem ser importadas e registra suas versões: PEFT 0.20.0 e Accelerate 1.14.0.

Guardar versões ajuda na reprodução porque APIs e comportamentos podem mudar entre lançamentos.

In [30]:
import peft
import accelerate

print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

PEFT: 0.20.0
Accelerate: 1.15.0


## 31. Carregamento do GPT-2 base

O modelo é carregado em `float16` na GPU. Essa precisão usa metade dos bits de `float32`, reduzindo memória, como escrever números com menos casas quando a precisão extra não é necessária.

Se a célula for repetida, referências anteriores e cache da GPU são liberados para evitar falta de memória. O modelo possui 124.439.808 parâmetros e ocupou cerca de 0,24 GB segundo o PyTorch.

**Avisos registrados:** o Windows não permitiu links simbólicos no cache; isso apenas pode consumir mais disco. A chave `masked_bias` apareceu como inesperada por diferença de versão/arquitetura e o próprio relatório indica que pode ser ignorada neste carregamento compatível; os pesos foram carregados completamente.

In [31]:
import gc
from transformers import AutoModelForCausalLM

# Se a célula for executada novamente, libera o modelo anterior.
if "modelo_base" in globals():
    del modelo_base
    gc.collect()
    torch.cuda.empty_cache()

dispositivo = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tipo_numerico = (
    torch.float16
    if dispositivo.type == "cuda"
    else torch.float32
)

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    dtype=tipo_numerico
)

modelo_base.config.pad_token_id = (
    tokenizador.pad_token_id
)

modelo_base.to(dispositivo)
modelo_base.eval()

total_parametros = sum(
    parametro.numel()
    for parametro in modelo_base.parameters()
)

print("Dispositivo:", next(modelo_base.parameters()).device)
print("Precisão:", next(modelo_base.parameters()).dtype)
print("Total de parâmetros:", f"{total_parametros:,}")

if dispositivo.type == "cuda":
    memoria = torch.cuda.memory_allocated() / 1024**3
    print("Memória ocupada no PyTorch:", f"{memoria:.2f} GB")

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 1264.46it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dispositivo: cuda:0
Precisão: torch.float16
Total de parâmetros: 124,439,808
Memória ocupada no PyTorch: 0.24 GB


## 32. Funções para gerar a resposta de referência

`formatar_prompt` cria a pergunta no mesmo padrão usado no treino, mas sem preencher a resposta. `gerar_resposta_base` pede ao GPT-2 original para completar esse espaço.

A geração é determinística (`do_sample=False`), limita-se a 80 novos tokens e aplica controles simples de repetição. Isso cria uma **baseline**, isto é, um ponto de comparação antes do ajuste fino.

In [32]:
def formatar_prompt(pergunta, contexto=""):
    partes = [
        f"### Pergunta:\n{pergunta.strip()}"
    ]

    if contexto.strip():
        partes.append(
            f"### Contexto:\n{contexto.strip()}"
        )

    partes.append("### Resposta:\n")

    return "\n\n".join(partes)


def gerar_resposta_base(pergunta, contexto=""):
    prompt = formatar_prompt(
        pergunta,
        contexto
    )

    entrada = tokenizador(
        prompt,
        return_tensors="pt"
    ).to(dispositivo)

    tamanho_entrada = entrada[
        "input_ids"
    ].shape[1]

    with torch.inference_mode():
        saida = modelo_base.generate(
            **entrada,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizador.eos_token_id,
            pad_token_id=tokenizador.pad_token_id
        )

    novos_tokens = saida[
        0,
        tamanho_entrada:
    ]

    resposta = tokenizador.decode(
        novos_tokens,
        skip_special_tokens=True
    ).strip()

    return resposta

## 33. Avaliação do modelo antes do treinamento

A pergunta fixa sobre fidelização é usada como régua comum para comparar as épocas.

**Resultado observado:** a resposta original é pouco útil e contraditória, afirmando que a empresa não teria razão para fidelizar clientes. Isso confirma que um modelo de linguagem geral ainda não está alinhado à tarefa específica do projeto.

In [33]:
pergunta_baseline = (
    "Como uma pequena empresa pode "
    "fidelizar seus clientes?"
)

resposta_baseline = gerar_resposta_base(
    pergunta_baseline
)

print("Pergunta:", pergunta_baseline)
print("\nResposta do GPT-2 original:")
print(resposta_baseline)

Pergunta: Como uma pequena empresa pode fidelizar seus clientes?

Resposta do GPT-2 original:
A resposta é que a empresa não tem nenhuma razão para fazer isso.

O que se sabe sobre o assunto, é que os consumidores são muito mais numerosos do que as empresas de tecnologia e podem ser facilmente confundidos com outras pessoas ou mesmo até mesmo um grupo de pessoas diferentes.
A empresa também possui vantagens em relação ao cliente, pois eles têm acesso aos serviços oferecidos por outros fornecedores (


## 34. Aplicação do LoRA

LoRA adiciona pequenas matrizes treináveis a partes do modelo e congela os pesos principais. Uma analogia é colocar “anotações especializadas” sobre um livro sem reescrever todas as páginas.

Foram treinados 294.912 parâmetros, apenas 0,2364% do total, o que reduz memória e tamanho do arquivo final.

**Aviso registrado:** a camada `c_attn` do GPT-2 usa `Conv1D`; por isso o PEFT ajustou automaticamente `fan_in_fan_out=True`. É uma correção de compatibilidade, não uma falha.

In [34]:
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

configuracao_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

modelo = get_peft_model(
    modelo_base,
    configuracao_lora
)

modelo.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


c:\Users\Pablo\Desktop\int-aprendizado-maquina\venv\Lib\site-packages\peft\tuners\lora\layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 35. Teste de passagem pelo modelo

Um único lote percorre o modelo sem atualizar pesos. Esse “teste de fumaça” verifica se dados, GPU, LoRA e cálculo da loss são compatíveis antes do laço completo.

**Resultado observado:** a loss 3,3912 é finita e as previsões têm formato `[2, 106, 50257]`: dois exemplos, 106 posições e uma pontuação para cada token do vocabulário.

**Aviso registrado:** `loss_type=None` não foi reconhecido, então o Transformers escolheu automaticamente `ForCausalLMLoss`, a loss padrão correta para previsão causal de texto.

In [35]:
# Pega somente um lote do treinamento.
lote_teste_modelo = next(
    iter(carregador_treino)
)

# Envia os três tensores do lote para a GPU.
lote_teste_modelo = {
    nome: tensor.to(dispositivo)
    for nome, tensor in lote_teste_modelo.items()
}

# Modo de avaliação: nenhum parâmetro será ajustado.
modelo.eval()

with torch.inference_mode():
    resultado_teste = modelo(
        **lote_teste_modelo
    )

print(
    "Loss do lote:",
    resultado_teste.loss.item()
)

print(
    "Formato das previsões:",
    resultado_teste.logits.shape
)

print(
    "Loss válida:",
    torch.isfinite(resultado_teste.loss).item()
)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss do lote: 3.8503897190093994
Formato das previsões: torch.Size([2, 133, 50257])
Loss válida: True


## 36. Configuração do otimizador

O AdamW decide como ajustar os parâmetros a partir dos erros calculados. A taxa de aprendizado `5e-4` controla o tamanho de cada correção — como o passo dado ao caminhar em direção a um ponto melhor.

Somente os 294.912 parâmetros do LoRA são entregues ao otimizador; os pesos originais permanecem congelados.

In [36]:
TAXA_APRENDIZADO = 5e-4

parametros_treinaveis = [
    parametro
    for parametro in modelo.parameters()
    if parametro.requires_grad
]

otimizador = torch.optim.AdamW(
    parametros_treinaveis,
    lr=TAXA_APRENDIZADO
)

quantidade_treinavel = sum(
    parametro.numel()
    for parametro in parametros_treinaveis
)

print(
    "Taxa de aprendizado:",
    TAXA_APRENDIZADO
)

print(
    "Tensores entregues ao otimizador:",
    len(parametros_treinaveis)
)

print(
    "Parâmetros entregues ao otimizador:",
    f"{quantidade_treinavel:,}"
)

Taxa de aprendizado: 0.0005
Tensores entregues ao otimizador: 24
Parâmetros entregues ao otimizador: 294,912


## 37. Função de avaliação pela loss

Esta função percorre a validação sem calcular gradientes e retorna a loss média. **Loss** é uma medida de erro: quanto menor, melhor o modelo prevê os próximos tokens dos textos esperados.

Ela não mede sozinha utilidade, correção factual ou qualidade de escrita. Por isso o notebook também compara respostas de forma qualitativa.

In [37]:
def calcular_loss_validacao(
    modelo,
    carregador,
    dispositivo
):
    modelo.eval()
    perdas = []

    with torch.inference_mode():
        for lote in carregador:
            lote_gpu = {
                nome: tensor.to(dispositivo)
                for nome, tensor in lote.items()
            }

            resultado = modelo(**lote_gpu)

            perdas.append(
                resultado.loss.item()
            )

    loss_media = sum(perdas) / len(perdas)

    return loss_media

## 38. Loss de validação antes do ajuste

Esta medição estabelece o ponto de partida do modelo com o LoRA ainda não treinado.

**Resultado observado:** loss inicial de validação igual a 3,5769. As épocas seguintes serão comparadas a esse valor para verificar se o modelo generaliza melhor para exemplos que não foram usados nos passos de treino.

In [38]:
loss_validacao_inicial = calcular_loss_validacao(
    modelo,
    carregador_validacao,
    dispositivo
)

print(
    "Loss inicial de validação:",
    round(loss_validacao_inicial, 4)
)

Loss inicial de validação: 3.5769


## 39. Primeira época de treinamento

Uma **época** é uma passagem completa pelos 960 exemplos, semelhante a revisar uma apostila inteira uma vez. Em cada lote o modelo prevê, calcula a loss, propaga o erro e o AdamW atualiza o LoRA.

**Resultado observado:** loss média de treino 2,8108 e loss de validação 2,5202, contra 3,5769 inicialmente. A queda sugere aprendizado útil também fora do treino. A época levou cerca de 0,43 minuto.

In [39]:
import time
from pathlib import Path
from tqdm import tqdm

EPOCAS = 1

historico_loss_passos = []
historico_epocas = []

modelo.config.use_cache = False
modelo.train()

inicio = time.time()
loss_total = 0

barra = tqdm(
    carregador_treino,
    desc="Época 1/1"
)

for passo, lote in enumerate(barra, start=1):
    # 1. Envia o lote para a GPU.
    lote_gpu = {
        nome: tensor.to(dispositivo)
        for nome, tensor in lote.items()
    }

    # 2. Apaga os gradientes do lote anterior.
    otimizador.zero_grad(
        set_to_none=True
    )

    # 3. O modelo faz as previsões e calcula a loss.
    resultado = modelo(**lote_gpu)
    loss = resultado.loss

    # 4. Calcula a direção das correções.
    loss.backward()

    # 5. O AdamW modifica os parâmetros do LoRA.
    otimizador.step()

    valor_loss = loss.item()
    loss_total += valor_loss
    historico_loss_passos.append(valor_loss)

    barra.set_postfix(
        loss=f"{valor_loss:.3f}"
    )

loss_treino_epoca = (
    loss_total / len(carregador_treino)
)

# Mede a validação depois da época.
loss_validacao_epoca = calcular_loss_validacao(
    modelo,
    carregador_validacao,
    dispositivo
)

duracao = time.time() - inicio

historico_epocas.append({
    "epoca": 1,
    "loss_treino": loss_treino_epoca,
    "loss_validacao": loss_validacao_epoca
})

print("\nTreinamento concluído!")
print(
    "Loss média de treino:",
    round(loss_treino_epoca, 4)
)
print(
    "Loss de validação:",
    round(loss_validacao_epoca, 4)
)
print(
    "Duração:",
    round(duracao / 60, 2),
    "minutos"
)

Época 1/1:   0%|          | 0/480 [00:00<?, ?it/s]

Época 1/1: 100%|██████████| 480/480 [00:46<00:00, 10.31it/s, loss=2.275]



Treinamento concluído!
Loss média de treino: 2.8063
Loss de validação: 2.5228
Duração: 0.84 minutos


## 40. Salvamento do checkpoint da época 1

Um **checkpoint** é um ponto de salvamento do experimento. São gravados o adaptador LoRA, o tokenizador, as métricas e o estado do otimizador, permitindo usar o modelo ou continuar o treinamento.

O adaptador tem cerca de 1,13 MB porque contém apenas os pesos adicionais do LoRA, não uma cópia completa do GPT-2. Para utilizá-lo depois, ainda será necessário carregar o mesmo modelo-base.

In [40]:
import json
from pathlib import Path

PASTA_ADAPTADOR = Path(
    "modelos/lora_negocios_epoca_1"
)

PASTA_ADAPTADOR.mkdir(
    parents=True,
    exist_ok=True
)

# Salva o adaptador LoRA.
modelo.save_pretrained(
    PASTA_ADAPTADOR
)

# Salva o tokenizador utilizado.
tokenizador.save_pretrained(
    PASTA_ADAPTADOR
)

# Guarda as decisões e os resultados do experimento.
metricas = {
    "modelo_base": MODELO_BASE,
    "epocas": 1,
    "batch_size": BATCH_SIZE,
    "learning_rate": TAXA_APRENDIZADO,
    "max_length": MAX_LENGTH,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "loss_validacao_inicial": loss_validacao_inicial,
    "loss_treino_epoca_1": loss_treino_epoca,
    "loss_validacao_epoca_1": loss_validacao_epoca,
    "duracao_segundos": duracao,
    "pergunta_baseline": pergunta_baseline,
    "resposta_baseline": resposta_baseline
}

with open(
    PASTA_ADAPTADOR / "metricas.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metricas,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

# Guarda o estado do otimizador caso desejemos
# continuar o treinamento posteriormente.
torch.save(
    {
        "epoca": 1,
        "optimizer_state_dict": (
            otimizador.state_dict()
        ),
        "historico_loss_passos": (
            historico_loss_passos
        ),
        "historico_epocas": (
            historico_epocas
        )
    },
    PASTA_ADAPTADOR / "estado_treinamento.pt"
)

print("Arquivos salvos em:", PASTA_ADAPTADOR.resolve())

for arquivo in PASTA_ADAPTADOR.iterdir():
    tamanho_mb = arquivo.stat().st_size / 1024**2

    print(
        f"- {arquivo.name}: "
        f"{tamanho_mb:.2f} MB"
    )

c:\Users\Pablo\Desktop\int-aprendizado-maquina\venv\Lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010) - silently ignoring the lookup for the file config.json in pierreguillou/gpt2-small-portuguese.
  warnings.warn(
c:\Users\Pablo\Desktop\int-aprendizado-maquina\venv\Lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in pierreguillou/gpt2-small-portuguese - will assume that the vocabulary was not modified.
  warnings.warn(


Arquivos salvos em: C:\Users\Pablo\Desktop\int-aprendizado-maquina\modelos\lora_negocios_epoca_1
- adapter_config.json: 0.00 MB
- adapter_model.safetensors: 1.13 MB
- avaliacao_qualitativa.csv: 0.00 MB
- estado_treinamento.pt: 2.27 MB
- metricas.json: 0.00 MB
- README.md: 0.00 MB
- tokenizer.json: 3.49 MB
- tokenizer_config.json: 0.00 MB


## 41. Função de geração com o modelo adaptado

Esta função reutiliza as mesmas regras determinísticas da baseline e aceita qualquer modelo de geração. Ela ativa o cache apenas durante a geração para acelerar a produção token a token.

Usar os mesmos parâmetros antes e depois isola melhor o efeito do treinamento: a principal diferença passa a ser o adaptador LoRA.

In [41]:
def gerar_resposta(
    modelo_geracao,
    pergunta,
    contexto=""
):
    prompt = formatar_prompt(
        pergunta,
        contexto
    )

    entrada = tokenizador(
        prompt,
        return_tensors="pt"
    ).to(dispositivo)

    tamanho_entrada = entrada[
        "input_ids"
    ].shape[1]

    modelo_geracao.eval()
    modelo_geracao.config.use_cache = True

    with torch.inference_mode():
        saida = modelo_geracao.generate(
            **entrada,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizador.eos_token_id,
            pad_token_id=tokenizador.pad_token_id
        )

    novos_tokens = saida[
        0,
        tamanho_entrada:
    ]

    return tokenizador.decode(
        novos_tokens,
        skip_special_tokens=True
    ).strip()

## 42. Comparação após a primeira época

A pergunta fixa é respondida novamente com o LoRA ativo.

**Resultado observado:** a resposta deixa de negar a importância da fidelização, mas continua vaga e pouco prática. Portanto, a melhora numérica da loss ainda não significa uma resposta satisfatória; esse é um exemplo da diferença entre avaliação quantitativa e qualitativa.

In [42]:
resposta_epoca_1 = gerar_resposta(
    modelo,
    pergunta_baseline
)

print("PERGUNTA")
print(pergunta_baseline)

print("\nANTES DO TREINAMENTO")
print(resposta_baseline)

print("\nDEPOIS DE UMA ÉPOCA")
print(resposta_epoca_1)

PERGUNTA
Como uma pequena empresa pode fidelizar seus clientes?

ANTES DO TREINAMENTO
A resposta é que a empresa não tem nenhuma razão para fazer isso.

O que se sabe sobre o assunto, é que os consumidores são muito mais numerosos do que as empresas de tecnologia e podem ser facilmente confundidos com outras pessoas ou mesmo até mesmo um grupo de pessoas diferentes.
A empresa também possui vantagens em relação ao cliente, pois eles têm acesso aos serviços oferecidos por outros fornecedores (

DEPOIS DE UMA ÉPOCA
A pequenas empresas podem fazer um investimento de US $ 500 por ano.


## 43. Avaliação qualitativa em cinco exemplos de validação

Cinco registros fixos são comparados em três colunas: resposta esperada, GPT-2 original e modelo após uma época. `disable_adapter()` desliga temporariamente o LoRA sem alterar seus pesos.

Essa amostra permite observar relevância, coerência e aderência ao formato. É útil, mas pequena; não deve ser tratada como uma avaliação conclusiva do modelo inteiro.

In [43]:
amostra_validacao = df_validacao.sample(
    5,
    random_state=42
)

resultados_avaliacao = []

# Gera respostas com o LoRA temporariamente desligado.
with modelo.disable_adapter():
    for indice, linha in amostra_validacao.iterrows():
        resposta_original = gerar_resposta(
            modelo,
            linha["instruction"],
            linha["input"]
        )

        resultados_avaliacao.append({
            "indice": indice,
            "pergunta": linha["instruction"],
            "contexto": linha["input"],
            "resposta_esperada": linha["output"],
            "antes": resposta_original
        })

# Gera respostas com o LoRA treinado ligado.
for resultado, (_, linha) in zip(
    resultados_avaliacao,
    amostra_validacao.iterrows()
):
    resultado["depois_epoca_1"] = gerar_resposta(
        modelo,
        linha["instruction"],
        linha["input"]
    )

df_avaliacao_qualitativa = pd.DataFrame(
    resultados_avaliacao
)

pd.set_option(
    "display.max_colwidth",
    None
)

display(
    df_avaliacao_qualitativa[
        [
            "indice",
            "pergunta",
            "resposta_esperada",
            "antes",
            "depois_epoca_1"
        ]
    ]
)

,indice,pergunta,resposta_esperada,antes,depois_epoca_1
0,233,Quais são algumas boas estratégias para usar ao negociar preços?,"Ao negociar preços, algumas boas estratégias a serem usadas incluem: determinar seu orçamento antes de começar a negociar, pesquisar a taxa de mercado para o item ou serviço, ficar calmo e respeitoso, entender as necessidades da outra parte, encontrar um terreno comum para se concentrar e apresentar alternativas aceitáveis. Também é importante estar sempre preparado para sair se você não puder chegar a um acordo.","A resposta é que a oferta não pode ser vendida, e sim negociada.\n\nO preço de um bem negociado é o preço do bem em questão (ou seja, da quantidade comprada).\n\nPara uma boa negociação, os valores dos bens negociáveis devem estar entre 1% e 5%.\n\nSe você quer comprar um bem, ele deve ter valor suficiente para satisfazer as necessidades básicas","As melhores ações que podem ser tomadas em negociações de preço incluem a compra ou venda do bem, o pagamento antecipado e os recursos necessários."
1,1004,Preveja se a cotação dada será bem sucedida no mercado.,positivo,"Requisitos de validação :\n\nA prova é feita por meio da seguinte fórmula:\n\nformula_1\n\nonde formula_2 representa o número de erros e formula_3 denota o erro.\nO algoritmo pode ser usado para calcular a média aritmética dos números inteiros positivos ou negativos, como em:\nformula-1\nformula–1\nOnde formula_4 representa o erro e","O resultado do processo de compra e venda é um aumento significativo na procura por produtos, serviços ou itens em geral para satisfazer as necessidades dos clientes; o preço da oferta também aumenta significativamente nos mercados consumidores mais amplos como os EUA (EUA) e Canadá onde há uma maior demanda crescente pelo produto interno bruto devido à alta taxa de crescimento econômico global."
2,745,Compor um e-mail para a equipe de marketing dizendo-lhes sobre um novo recurso que foi adicionado ao produto.,"Querida equipe de marketing, estamos entusiasmados em anunciar que agora adicionamos um novo recurso ao nosso produto! Este novo recurso foi projetado para melhorar a experiência do usuário e tornar ainda mais fácil para eles realizar suas tarefas. Esperamos que ele tenha um grande impacto na satisfação do cliente e é algo que achamos que nossos usuários vão adorar. Obrigado por seu suporte contínuo, equipe XYZ","Requisitos da equipe do marketing são enviados à equipe de desenvolvimento, onde eles podem ser usados ​​para criar uma nova aplicação ou melhorar o desempenho do produto. Os desenvolvedores também devem fornecer feedback positivo sobre o novo recurso.\nA empresa pode então enviar os resultados dos testes com uma mensagem simples como ""Este recurso é ótimo"".\n\nO processo de seleção de recursos na web é semelhante aos processos de","Um serviço semelhante é chamado ""Stop"" ou ""smartphone"". O objetivo do site, no entanto, era criar uma experiência mais interativa com o uso da tecnologia Smartphones em seus dispositivos móveis sem fio (como smartphones)."
3,461,Desenhe um plano para otimizar o site da empresa.,"Um plano abrangente para otimizar o site da empresa inclui realizar uma auditoria de SEO, melhorar o conteúdo do site, otimizar imagens e vídeos, otimizar a velocidade do site, usar software de rastreamento de análise e campanhas de marketing.","O site é projetado para ser uma plataforma de busca e desenvolvimento, onde os usuários podem criar perfis pessoais ou postar em seu perfil no Facebook, Twitter, Instagram, ou qualquer outro website que ofereça serviços como ""smartphones"" (como o Google Maps) ou ""tablets"".\n\nA plataforma permite aos usuários adicionar novos perfis a seus amigos através do Facebook, por meio dos perfis","O objetivo do site é melhorar a qualidade de atendimento e oferecer uma experiência mais agradável ao cliente, oferecendo serviços personalizados que atendem às necessidades dos clientes em geral ou com base no feedback positivo das pessoas envol

## 44. Exportação da avaliação da época 1

A tabela comparativa é salva em CSV dentro da pasta do checkpoint. O formato `utf-8-sig` facilita a abertura de textos acentuados no Excel em Windows.

O arquivo cria um registro permanente da análise, separado da visualização do notebook.

In [44]:
df_avaliacao_qualitativa.to_csv(
    PASTA_ADAPTADOR / "avaliacao_qualitativa.csv",
    index=False,
    encoding="utf-8-sig"
)

## 45. Função reutilizável para novas épocas

O laço de treinamento da primeira época é encapsulado em `treinar_uma_epoca`. Isso evita copiar a mesma lógica e reduz o risco de executar etapas diferentes em épocas posteriores.

A função retorna loss média, loss de cada passo e duração. A validação continua fora dela para manter treino e avaliação explicitamente separados.

In [45]:
def treinar_uma_epoca(
    modelo,
    carregador,
    otimizador,
    dispositivo,
    numero_epoca
):
    modelo.config.use_cache = False
    modelo.train()

    losses = []
    inicio = time.time()

    barra = tqdm(
        carregador,
        desc=f"Época {numero_epoca}"
    )

    for lote in barra:
        lote_gpu = {
            nome: tensor.to(dispositivo)
            for nome, tensor in lote.items()
        }

        otimizador.zero_grad(
            set_to_none=True
        )

        resultado = modelo(**lote_gpu)
        loss = resultado.loss

        loss.backward()
        otimizador.step()

        valor_loss = loss.item()
        losses.append(valor_loss)

        barra.set_postfix(
            loss=f"{valor_loss:.3f}"
        )

    loss_media = sum(losses) / len(losses)
    duracao = time.time() - inicio

    return loss_media, losses, duracao

## 46. Segunda época de treinamento

O modelo percorre novamente os dados, preservando os pesos do LoRA e o estado do otimizador da época anterior.

**Resultado observado:** loss de treino 2,6394 e validação 2,4826. A validação caiu 0,0376 em relação à época 1, uma melhora menor que a primeira, mas ainda na direção correta. Não há sinal numérico de overfitting até aqui.

In [46]:
loss_treino_epoca_2, losses_epoca_2, duracao_epoca_2 = (
    treinar_uma_epoca(
        modelo,
        carregador_treino,
        otimizador,
        dispositivo,
        numero_epoca=2
    )
)

loss_validacao_epoca_2 = calcular_loss_validacao(
    modelo,
    carregador_validacao,
    dispositivo
)

historico_loss_passos.extend(
    losses_epoca_2
)

historico_epocas.append({
    "epoca": 2,
    "loss_treino": loss_treino_epoca_2,
    "loss_validacao": loss_validacao_epoca_2
})

print("\nÉpoca 2 concluída!")
print(
    "Loss média de treino:",
    round(loss_treino_epoca_2, 4)
)
print(
    "Loss de validação:",
    round(loss_validacao_epoca_2, 4)
)
print(
    "Validação na época 1:",
    round(loss_validacao_epoca, 4)
)
print(
    "Duração:",
    round(duracao_epoca_2 / 60, 2),
    "minutos"
)

Época 2: 100%|██████████| 480/480 [00:58<00:00,  8.26it/s, loss=2.300]



Época 2 concluída!
Loss média de treino: 2.6414
Loss de validação: 2.4724
Validação na época 1: 2.5228
Duração: 0.97 minutos


## 47. Salvamento do checkpoint da época 2

O segundo adaptador, suas métricas acumuladas e o estado do otimizador são gravados em uma pasta própria.

Separar checkpoints permite voltar a uma época anterior caso treinamentos adicionais piorem a validação ou a qualidade das respostas.

In [47]:
PASTA_EPOCA_2 = Path(
    "modelos/lora_negocios_epoca_2"
)

PASTA_EPOCA_2.mkdir(
    parents=True,
    exist_ok=True
)

modelo.save_pretrained(PASTA_EPOCA_2)
tokenizador.save_pretrained(PASTA_EPOCA_2)

metricas_epoca_2 = {
    "modelo_base": MODELO_BASE,
    "epocas_acumuladas": 2,
    "batch_size": BATCH_SIZE,
    "learning_rate": TAXA_APRENDIZADO,
    "max_length": MAX_LENGTH,
    "loss_validacao_inicial": loss_validacao_inicial,
    "loss_treino_epoca_1": loss_treino_epoca,
    "loss_validacao_epoca_1": loss_validacao_epoca,
    "loss_treino_epoca_2": loss_treino_epoca_2,
    "loss_validacao_epoca_2": loss_validacao_epoca_2,
    "duracao_epoca_1": duracao,
    "duracao_epoca_2": duracao_epoca_2
}

with open(
    PASTA_EPOCA_2 / "metricas.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metricas_epoca_2,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

torch.save(
    {
        "epoca": 2,
        "optimizer_state_dict": (
            otimizador.state_dict()
        ),
        "historico_loss_passos": (
            historico_loss_passos
        ),
        "historico_epocas": (
            historico_epocas
        )
    },
    PASTA_EPOCA_2 / "estado_treinamento.pt"
)

print(
    "Época 2 salva em:",
    PASTA_EPOCA_2.resolve()
)

Época 2 salva em: C:\Users\Pablo\Desktop\int-aprendizado-maquina\modelos\lora_negocios_epoca_2


## 48. Comparação qualitativa após a época 2

Os mesmos cinco exemplos são gerados novamente, adicionados ao CSV e exibidos lado a lado com os resultados da época 1.

Manter a amostra fixa torna a comparação justa. Ainda assim, a escolha da melhor época deve considerar tanto essas respostas quanto a loss de validação e, idealmente, uma avaliação maior.

In [48]:
respostas_epoca_2 = []

for _, linha in amostra_validacao.iterrows():
    resposta = gerar_resposta(
        modelo,
        linha["instruction"],
        linha["input"]
    )

    respostas_epoca_2.append(resposta)

df_avaliacao_qualitativa[
    "depois_epoca_2"
] = respostas_epoca_2

df_avaliacao_qualitativa.to_csv(
    PASTA_EPOCA_2 / "avaliacao_qualitativa.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    df_avaliacao_qualitativa[
        [
            "indice",
            "pergunta",
            "resposta_esperada",
            "depois_epoca_1",
            "depois_epoca_2"
        ]
    ]
)

,indice,pergunta,resposta_esperada,depois_epoca_1,depois_epoca_2
0,233,Quais são algumas boas estratégias para usar ao negociar preços?,"Ao negociar preços, algumas boas estratégias a serem usadas incluem: determinar seu orçamento antes de começar a negociar, pesquisar a taxa de mercado para o item ou serviço, ficar calmo e respeitoso, entender as necessidades da outra parte, encontrar um terreno comum para se concentrar e apresentar alternativas aceitáveis. Também é importante estar sempre preparado para sair se você não puder chegar a um acordo.","As melhores ações que podem ser tomadas em negociações de preço incluem a compra ou venda do bem, o pagamento antecipado e os recursos necessários.","As melhores ações de negociação do mercado estão em negociações com clientes, fornecedores e parceiros. As empresas podem oferecer uma variedade variada ofertas que incluem produtos ou serviços variados como jogos eletrônicos; aplicativos comerciais voltados a negócios corporativos (como o serviço da marca); publicidade corporativo direcionada à indústria automotiva comercial tradicional - incluindo anúncios on-line relacionados aos mercados emergentes – além disso oferecendo um amplo leque de opções oferecidas por meio dos"
1,1004,Preveja se a cotação dada será bem sucedida no mercado.,positivo,"O resultado do processo de compra e venda é um aumento significativo na procura por produtos, serviços ou itens em geral para satisfazer as necessidades dos clientes; o preço da oferta também aumenta significativamente nos mercados consumidores mais amplos como os EUA (EUA) e Canadá onde há uma maior demanda crescente pelo produto interno bruto devido à alta taxa de crescimento econômico global.","A alteração é muito boa, mas não significa necessariamente uma melhoria em qualidade de vida ou um aumento na produtividade do produto.”"
2,745,Compor um e-mail para a equipe de marketing dizendo-lhes sobre um novo recurso que foi adicionado ao produto.,"Querida equipe de marketing, estamos entusiasmados em anunciar que agora adicionamos um novo recurso ao nosso produto! Este novo recurso foi projetado para melhorar a experiência do usuário e tornar ainda mais fácil para eles realizar suas tarefas. Esperamos que ele tenha um grande impacto na satisfação do cliente e é algo que achamos que nossos usuários vão adorar. Obrigado por seu suporte contínuo, equipe XYZ","Um serviço semelhante é chamado ""Stop"" ou ""smartphone"". O objetivo do site, no entanto, era criar uma experiência mais interativa com o uso da tecnologia Smartphones em seus dispositivos móveis sem fio (como smartphones).","Um serviço oferecido por uma empresa pode ser fornecido gratuitamente ou grátis, oferecendo serviços como compras online com preços baixos em lojas virtuais (como Amazon). O conteúdo do site também permite aos usuários criar perfis personalizados usando o perfil desejado no aplicativo oficial da empresa; os dados podem ajudar na busca dos clientes mais bem pagos pela compra direta sem precisar comprar qualquer item específico diretamente dele - esses são alguns exemplos incluem anúncios"
3,461,Desenhe um plano para otimizar o site da empresa.,"Um plano abrangente para otimizar o site da empresa inclui realizar uma auditoria de SEO, melhorar o conteúdo do site, otimizar imagens e vídeos, otimizar a velocidade do site, usar software de rastreamento de análise e campanhas de marketing.","O objetivo do site é melhorar a qualidade de atendimento e oferecer uma experiência mais agradável ao cliente, oferecendo serviços personalizados que atendem às necessidades dos clientes em geral ou com base no feedback positivo das pessoas envolvidas na organização social corporativa (e também fornecendo informações sobre os produtos oferecidos).","O objetivo do marketing viral é criar uma imagem positiva e atraente que possa ser vista por todos os públicos, incluindo aqueles com necessidades especiais ou de alta renda social; também pode ajudar a promover produtos relacionados à marca

## 49. Terceira época de treinamento

O treinamento continua a partir da época 2, sem reinicializar modelo ou otimizador.

**Resultado observado:** loss de treino 2,5825 e validação 2,4269. A validação caiu mais 0,0557 e atingiu o melhor valor registrado. Como treino e validação melhoraram juntos, não há evidência numérica de overfitting nas três épocas, embora mais épocas exijam monitoramento.

In [49]:
loss_treino_epoca_3, losses_epoca_3, duracao_epoca_3 = (
    treinar_uma_epoca(
        modelo,
        carregador_treino,
        otimizador,
        dispositivo,
        numero_epoca=3
    )
)

loss_validacao_epoca_3 = calcular_loss_validacao(
    modelo,
    carregador_validacao,
    dispositivo
)

historico_loss_passos.extend(
    losses_epoca_3
)

historico_epocas.append({
    "epoca": 3,
    "loss_treino": loss_treino_epoca_3,
    "loss_validacao": loss_validacao_epoca_3
})

print("\nÉpoca 3 concluída!")
print(
    "Loss média de treino:",
    round(loss_treino_epoca_3, 4)
)
print(
    "Loss de validação:",
    round(loss_validacao_epoca_3, 4)
)
print(
    "Validação anterior:",
    round(loss_validacao_epoca_2, 4)
)
print(
    "Duração:",
    round(duracao_epoca_3 / 60, 2),
    "minutos"
)

Época 3: 100%|██████████| 480/480 [00:38<00:00, 12.63it/s, loss=2.128]



Época 3 concluída!
Loss média de treino: 2.5709
Loss de validação: 2.4397
Validação anterior: 2.4724
Duração: 0.63 minutos


## 50. Salvamento do checkpoint da época 3

Esta célula salva o adaptador correspondente ao melhor valor de validação observado, além das métricas das três épocas e do estado completo necessário para continuidade.

Manter `modelo_base`, `max_length`, configuração do LoRA e demais hiperparâmetros registrados é essencial para reproduzir o carregamento futuro.

In [50]:
PASTA_EPOCA_3 = Path(
    "modelos/lora_negocios_epoca_3"
)

PASTA_EPOCA_3.mkdir(
    parents=True,
    exist_ok=True
)

modelo.save_pretrained(PASTA_EPOCA_3)
tokenizador.save_pretrained(PASTA_EPOCA_3)

metricas_epoca_3 = {
    "modelo_base": MODELO_BASE,
    "epocas_acumuladas": 3,
    "batch_size": BATCH_SIZE,
    "learning_rate": TAXA_APRENDIZADO,
    "max_length": MAX_LENGTH,
    "loss_validacao_inicial": loss_validacao_inicial,
    "loss_treino_epoca_1": loss_treino_epoca,
    "loss_validacao_epoca_1": loss_validacao_epoca,
    "loss_treino_epoca_2": loss_treino_epoca_2,
    "loss_validacao_epoca_2": loss_validacao_epoca_2,
    "loss_treino_epoca_3": loss_treino_epoca_3,
    "loss_validacao_epoca_3": loss_validacao_epoca_3
}

with open(
    PASTA_EPOCA_3 / "metricas.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metricas_epoca_3,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

torch.save(
    {
        "epoca": 3,
        "optimizer_state_dict": otimizador.state_dict(),
        "historico_loss_passos": historico_loss_passos,
        "historico_epocas": historico_epocas
    },
    PASTA_EPOCA_3 / "estado_treinamento.pt"
)

print("Época 3 salva em:", PASTA_EPOCA_3.resolve())

Época 3 salva em: C:\Users\Pablo\Desktop\int-aprendizado-maquina\modelos\lora_negocios_epoca_3


## 51. Comparação qualitativa após a época 3

As respostas da terceira época são adicionadas à mesma tabela e comparadas às da segunda. Isso ajuda a verificar se a melhora da loss também aparece como respostas mais úteis e coerentes.

O CSV final é salvo junto ao checkpoint da época 3 para manter resultado e modelo associados.

In [51]:
respostas_epoca_3 = []

for _, linha in amostra_validacao.iterrows():
    respostas_epoca_3.append(
        gerar_resposta(
            modelo,
            linha["instruction"],
            linha["input"]
        )
    )

df_avaliacao_qualitativa[
    "depois_epoca_3"
] = respostas_epoca_3

df_avaliacao_qualitativa.to_csv(
    PASTA_EPOCA_3 / "avaliacao_qualitativa.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    df_avaliacao_qualitativa[
        [
            "indice",
            "pergunta",
            "resposta_esperada",
            "depois_epoca_2",
            "depois_epoca_3"
        ]
    ]
)

,indice,pergunta,resposta_esperada,depois_epoca_2,depois_epoca_3
0,233,Quais são algumas boas estratégias para usar ao negociar preços?,"Ao negociar preços, algumas boas estratégias a serem usadas incluem: determinar seu orçamento antes de começar a negociar, pesquisar a taxa de mercado para o item ou serviço, ficar calmo e respeitoso, entender as necessidades da outra parte, encontrar um terreno comum para se concentrar e apresentar alternativas aceitáveis. Também é importante estar sempre preparado para sair se você não puder chegar a um acordo.","As melhores ações de negociação do mercado estão em negociações com clientes, fornecedores e parceiros. As empresas podem oferecer uma variedade variada ofertas que incluem produtos ou serviços variados como jogos eletrônicos; aplicativos comerciais voltados a negócios corporativos (como o serviço da marca); publicidade corporativo direcionada à indústria automotiva comercial tradicional - incluindo anúncios on-line relacionados aos mercados emergentes – além disso oferecendo um amplo leque de opções oferecidas por meio dos","As melhores ações de negociação do mercado estão em oferecer uma variedade variada ofertas, incluindo negociações com clientes e fornecedores. As empresas podem fornecer um serviço gratuito ou grátis que permite aos seus funcionários a comprarem produtos selecionados por meio da venda online sem precisar pagar taxas adicionais (por exemplo vendendo itens exclusivos). Além disso as equipes devem garantir o atendimento adequado às necessidades dos consumidores durante os meses finais antes das vendas serem realizadas"
1,1004,Preveja se a cotação dada será bem sucedida no mercado.,positivo,"A alteração é muito boa, mas não significa necessariamente uma melhoria em qualidade de vida ou um aumento na produtividade do produto.”","O resultado do desenvolvimento da tecnologia de comunicação pode ser bom, mas o custo é alto e os dados são difíceis para serem processados por equipes envolvidas em tarefas específicas ou com uma equipe especializada na tarefa exata — especialmente quando as empresas estão tentando encontrar soluções inovadoras ao problema específico dos clientes mais comuns à empresa (por exemplo)."
2,745,Compor um e-mail para a equipe de marketing dizendo-lhes sobre um novo recurso que foi adicionado ao produto.,"Querida equipe de marketing, estamos entusiasmados em anunciar que agora adicionamos um novo recurso ao nosso produto! Este novo recurso foi projetado para melhorar a experiência do usuário e tornar ainda mais fácil para eles realizar suas tarefas. Esperamos que ele tenha um grande impacto na satisfação do cliente e é algo que achamos que nossos usuários vão adorar. Obrigado por seu suporte contínuo, equipe XYZ","Um serviço oferecido por uma empresa pode ser fornecido gratuitamente ou grátis, oferecendo serviços como compras online com preços baixos em lojas virtuais (como Amazon). O conteúdo do site também permite aos usuários criar perfis personalizados usando o perfil desejado no aplicativo oficial da empresa; os dados podem ajudar na busca dos clientes mais bem pagos pela compra direta sem precisar comprar qualquer item específico diretamente dele - esses são alguns exemplos incluem anúncios","O aplicativo permite aos clientes criar perfis personalizados, enviar mensagens ou receber avisos quando estão em uma situação difícil com o objetivo do cliente escolherem se tornar mais eficaz no atendimento à empresa oferecendo serviços gratuitos às suas necessidades diárias (como compras). Além disso os usuários podem adicionar anúncios relacionados diretamente na página da loja através dos links neles enviados por telefone celular sem precisar fazer qualquer tipo de ligação direta entre eles sozinhos;"
3,461,Desenhe um plano para otimizar o site da empresa.,"Um plano abrangente para otimizar o site da empresa inclui realizar uma auditoria de SEO, melhorar o conteúdo do site, otimizar imagens e vídeos, otimizar a velocidade do site, usar software de

## 52. Teste final com a pergunta de referência

O melhor checkpoint observado responde novamente à pergunta usada desde a baseline.

**Resultado observado:** a resposta passou a sugerir marketing, publicidade e adequação do produto, portanto está mais relacionada ao domínio. Contudo, ainda apresenta problemas de concordância, repetição e falta de ações diretamente ligadas à fidelização. O ajuste aprendeu parte do tema, mas o projeto ainda se beneficiaria de mais dados de alta qualidade, avaliação mais ampla e teste de hiperparâmetros.

In [ ]:
resposta_epoca_3 = gerar_resposta(
    modelo,
    pergunta_baseline
)

print(resposta_epoca_3)

A pequenas empresas podem fazer um investimento em seu produto ou serviço. Eles também devem fornecer serviços de apoio e aconselhamento para os funcionários, que são treinados sobre como a tecnologia deve ser usada no atendimento ao cliente; ajudar com o gerenciamento do tempo perdido por causa da falta crônica dos produtos/serviços perdidos (e outros problemas relacionados à manutenção); oferecer suporte técnico aos colaboradores regulares durante as sessões legais das atividades realizadas


: 

# Conclusões, avisos e limitações registradas

## Evolução observada

| Momento | Loss de validação |
|---|---:|
| Antes do treino | 3,5769 |
| Época 1 | 2,5202 |
| Época 2 | 2,4826 |
| Época 3 | 2,4269 |

A queda contínua indica que o LoRA aprendeu padrões que também aparecem na validação. A terceira época foi a melhor entre as executadas. Isso não prova, sozinho, que as respostas sejam corretas ou úteis: a avaliação qualitativa mostrou melhora de tema, mas ainda revelou linguagem pouco natural e recomendações vagas.

## Erros e conflitos

Não há `Traceback`, exceção ou célula interrompida registrada nas saídas atuais. Os eventos encontrados foram **avisos não fatais**:

- `IProgress not found`: afeta somente a aparência da barra de progresso no Jupyter; instalar/atualizar `ipywidgets` é opcional.
- Requisições sem `HF_TOKEN`: podem ser mais lentas ou sofrer limite de acesso; não modificam o modelo baixado.
- Cache sem links simbólicos no Windows: usa mais espaço em disco, mas continua funcional. O Modo de Desenvolvedor do Windows pode resolver.
- `masked_bias` inesperado: diferença de compatibilidade ao carregar o checkpoint; o carregamento terminou e o próprio Transformers classificou a chave como ignorável neste caso.
- Ajuste `fan_in_fan_out=True`: correção automática do PEFT porque `c_attn` é uma camada `Conv1D` do GPT-2.
- `loss_type=None`: o Transformers escolheu `ForCausalLMLoss`, que é a loss padrão apropriada para este modelo causal.

## Limitações e próximos passos

- A base tem somente 1.200 exemplos e 79,08% não possuem contexto; ampliar e revisar a qualidade dos dados pode ajudar mais do que apenas aumentar épocas.
- A avaliação qualitativa usa cinco exemplos. Uma amostra maior, com critérios objetivos, dará uma conclusão mais confiável.
- O conjunto de teste foi separado corretamente, mas ainda não foi usado. Ele deve ser avaliado somente após escolher definitivamente o checkpoint e os hiperparâmetros.
- Como as respostas finais ainda têm problemas linguísticos, vale testar taxa de aprendizado menor, mais dados e critérios de parada antecipada antes de considerar o modelo pronto para uso.